In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

from xgboost import XGBClassifier

Mounted at /content/drive


In [2]:
DATA_PATH = "/content/drive/MyDrive/datasets/raw/"

In [3]:
df = pd.read_csv(DATA_PATH + "application_train.csv")
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# DAYS_EMPLOYED anomaly
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# EXT_SOURCE missingness (only where EDA showed it's predictive)
df["EXT_SOURCE_1_MISSING"] = df["EXT_SOURCE_1"].isna().astype(int)
df["EXT_SOURCE_3_MISSING"] = df["EXT_SOURCE_3"].isna().astype(int)

In [5]:
print("DAYS_EMPLOYED anomaly count:", df["DAYS_EMPLOYED_ANOM"].sum())
print("Remaining 365243 values:", (df["DAYS_EMPLOYED"] == 365243).sum())
print("EXT_SOURCE_1 missing count:", df["EXT_SOURCE_1_MISSING"].sum())
print("EXT_SOURCE_3 missing count:", df["EXT_SOURCE_3_MISSING"].sum())

DAYS_EMPLOYED anomaly count: 55374
Remaining 365243 values: 0
EXT_SOURCE_1 missing count: 173378
EXT_SOURCE_3 missing count: 60965


In [6]:
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
y = df["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 123)
y shape: (307511,)


In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target %:\n", y_train.value_counts(normalize=True) * 100)
print("\nValid target %:\n", y_valid.value_counts(normalize=True) * 100)

Training shape: (246008, 123)
Validation shape: (61503, 123)

Train target %:
 TARGET
0    91.927092
1     8.072908
Name: proportion, dtype: float64

Valid target %:
 TARGET
0    91.927223
1     8.072777
Name: proportion, dtype: float64


In [9]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 107
Categorical features: 16


In [10]:
numeric_transformer_lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_lr = ColumnTransformer(transformers=[
    ("num", numeric_transformer_lr, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [11]:
numeric_transformer_xgb = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor_xgb = ColumnTransformer(transformers=[
    ("num", numeric_transformer_xgb, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [12]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_lr),
    ("model", lr_model)
])

In [13]:
print("Training Logistic Regression...")
lr_pipeline.fit(X_train, y_train)
print("Logistic Regression training complete.")

Training Logistic Regression...
Logistic Regression training complete.


In [14]:
lr_valid_proba = lr_pipeline.predict_proba(X_valid)[:, 1]

In [15]:
lr_roc_auc = roc_auc_score(y_valid, lr_valid_proba)
lr_pr_auc = average_precision_score(y_valid, lr_valid_proba)

print("LOGISTIC REGRESSION BASELINE")
print(f"ROC-AUC: {lr_roc_auc:.4f}")
print(f"PR-AUC:  {lr_pr_auc:.4f}")

LOGISTIC REGRESSION BASELINE
ROC-AUC: 0.7501
PR-AUC:  0.2326


In [16]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_xgb),
    ("model", xgb_model)
])

In [17]:
print("Training XGBoost baseline...")
xgb_pipeline.fit(X_train, y_train)
print("XGBoost baseline training complete.")

Training XGBoost baseline...
XGBoost baseline training complete.


In [18]:
xgb_valid_proba = xgb_pipeline.predict_proba(X_valid)[:, 1]

In [19]:
xgb_roc_auc = roc_auc_score(y_valid, xgb_valid_proba)
xgb_pr_auc = average_precision_score(y_valid, xgb_valid_proba)

print("XGBOOST BASELINE")
print(f"ROC-AUC: {xgb_roc_auc:.4f}")
print(f"PR-AUC:  {xgb_pr_auc:.4f}")

XGBOOST BASELINE
ROC-AUC: 0.7612
PR-AUC:  0.2516


In [20]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 11.38710976837865


In [21]:
# using scale pos weights
xgb_model_weighted = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

xgb_pipeline_weighted = Pipeline(steps=[
    ("preprocessor", preprocessor_xgb),
    ("model", xgb_model_weighted)
])

In [22]:
print("Training XGBoost with scale_pos_weight...")
xgb_pipeline_weighted.fit(X_train, y_train)
print("Weighted XGBoost training complete.")

Training XGBoost with scale_pos_weight...
Weighted XGBoost training complete.


In [23]:
xgb_w_valid_proba = xgb_pipeline_weighted.predict_proba(X_valid)[:, 1]

In [24]:
xgb_w_roc_auc = roc_auc_score(y_valid, xgb_w_valid_proba)
xgb_w_pr_auc = average_precision_score(y_valid, xgb_w_valid_proba)

print("XGBOOST + SCALE_POS_WEIGHT")
print(f"ROC-AUC: {xgb_w_roc_auc:.4f}")
print(f"PR-AUC:  {xgb_w_pr_auc:.4f}")

XGBOOST + SCALE_POS_WEIGHT
ROC-AUC: 0.7600
PR-AUC:  0.2493


In [25]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost", "XGBoost + scale_pos_weight"],
    "ROC-AUC": [lr_roc_auc, xgb_roc_auc, xgb_w_roc_auc],
    "PR-AUC": [lr_pr_auc, xgb_pr_auc, xgb_w_pr_auc]
})

display(results.style.format({"ROC-AUC": "{:.4f}", "PR-AUC": "{:.4f}"}))

,Model,ROC-AUC,PR-AUC
0,Logistic Regression,0.7501,0.2326
1,XGBoost,0.7612,0.2516
2,XGBoost + scale_pos_weight,0.7600,0.2493


In [26]:
print("XGBoost ROC-AUC improvement:", round(xgb_roc_auc - lr_roc_auc, 4))
print("XGBoost PR-AUC improvement:", round(xgb_pr_auc - lr_pr_auc, 4))

XGBoost ROC-AUC improvement: 0.0111
XGBoost PR-AUC improvement: 0.019


In [27]:
print("ROC-AUC change:", round(xgb_w_roc_auc - xgb_roc_auc, 4))
print("PR-AUC change:", round(xgb_w_pr_auc - xgb_pr_auc, 4))

ROC-AUC change: -0.0012
PR-AUC change: -0.0023


In [28]:
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"
os.makedirs(RESULTS_PATH, exist_ok=True)

results.to_csv(RESULTS_PATH + "baseline_results.csv", index=False)
print("Baseline results saved to:", RESULTS_PATH + "baseline_results.csv")

Baseline results saved to: /content/drive/MyDrive/RupeeRisk/baseline_results.csv


In [29]:
split_ids = pd.DataFrame({"SK_ID_CURR": df.loc[X_train.index, "SK_ID_CURR"], "split": "train"})
valid_ids = pd.DataFrame({"SK_ID_CURR": df.loc[X_valid.index, "SK_ID_CURR"], "split": "valid"})
split_ids = pd.concat([split_ids, valid_ids], ignore_index=True)

split_ids.to_csv(RESULTS_PATH + "train_valid_split.csv", index=False)
print("Train/validation split saved.")

Train/validation split saved.
